## What is RAG?



![LLM On Unseen data](data/llm_question_ansering.png)








## Full RAG Pipeline

Complete Retrieval-Augmented Generation with LangChain Expresiion Language (LCEL).

In [2]:
# Import necessary libraries for environment variable management and OpenAI API interaction
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv()) # read and set environment variables fromlocal .env file

In [3]:
# Create a document containing information about channel
%mkdir -p data
with open('data/breakdown.txt', 'w') as f:
    f.write("Welcome to BreakDown!\n" +
             "We simplify complex technologies through clear explanations, visual learning, and practical implementations across these areas:\n" +
            "- Latest research in AI, machine learning, and deep learning models\n" +
            "- Research-paper breakdowns and practical coding tutorials\n" +
            "- Applications of AI in autonomous driving, ADAS, and robotics\n" +
            "- Data structures, algorithms, and problem-solving techniques\n" +
            "Subscribe to explore cutting-edge technologies.\n" +
            "Maintained by Mashhood Raza and Aamna Hasan.\n")

### 1. Load the documents and create a vector store for RAG (Retrieval-Augmented Generation) tasks.
#### Load → Split → Embed & Store

#### Step 1 — Load documents
LangChain provides loaders for dozens of file formats. 
They all return a list of Document objects with metadata and page_content.

In [4]:
# Import module for loading text files 
from langchain_community.document_loaders import TextLoader

# Loads the text file and creates a list of documents.
loader= TextLoader("./data/breakdown.txt") #each document has metadata (source, page number, etc.) and page_content (the text).
docs = loader.load() # Load the documents from the text file

print(len(docs)) # Display the number of documents loaded
print(docs[0].metadata) # Display the metadata of the first document
print(docs[0].page_content[:200]) # Display the first 200 characters of the first document

1
{'source': './data/breakdown.txt'}
Welcome to BreakDown!
We simplify complex technologies through clear explanations, visual learning, and practical implementations across these areas:
- Latest research in AI, machine learning, and dee


#### Step 2 — Chunk the text
LLMs have context limits, and embeddings work best on focused passages — not entire documents. You split documents into smaller chunks before embedding. There are different chunking strategies that can be used.

In [5]:
# Import module for splitting the documents into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Splits the documents into smaller chunks based on the specified chunk size and overlap.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,       # characters per chunk
    chunk_overlap=20,     # overlap to preserve context across chunk boundaries
)
chunks = splitter.split_documents(docs) 

print(len(chunks)) # Display the total number of chunks
print(chunks[0].metadata) # Display the metadata of the first chunk
print(chunks[0].page_content[:50]) # Display the first 50 characters of the first chunk


8
{'source': './data/breakdown.txt'}
Welcome to BreakDown!


#### Step 3 — Embed and store
An embedding model converts each text chunk into a numeric vector. Similar chunks produce similar vectors — this is what enables semantic search.

FAISS (Facebook AI Similarity Search) is an open-source library built by Meta's Fundamental AI Research (FAIR) group. It is specifically designed for efficient similarity search and clustering of dense vectors.

FAISS vs Chroma vs Pinecone: FAISS is great for local development (no server). Chroma is easy to set up with persistence. Pinecone / Weaviate / Qdrant are cloud-hosted for production scale.

In [ ]:
# Import modules for creating embeddings and storing them in a vector store
from langchain.embeddings import HuggingFaceEmbeddings
#from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Instantiate the embeddings model with the specified model name
embeddings = HuggingFaceEmbeddings(model_name="nomic-ai/nomic-embed-text-v1.5", model_kwargs = {'trust_remote_code': True})
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Embed all chunks and store in FAISS (in-memory, no server needed)
vector_store = FAISS.from_documents(chunks, embeddings)

# Save to disk so you don't re-embed every time
vector_store.save_local("faiss_index")

<All keys matched successfully>


### 2. Load and Retrieve the vector store from disk, write prompt, and perform RAG (Retrieval-Augmented Generation) tasks

#### Load & Retrieve → Prompt → RAG Chain

#### Step 1 — Load and Retrieve
A retriever takes a query string and returns the most relevant chunks. The default is similarity search (cosine distance between embeddings).

__MMR — Maximum Marginal Relevance:__ Instead of returning the top-k most similar chunks (which may all say the same thing), MMR balances relevance with diversity. Use search_type="mmr" when your chunks have redundant content.MMR fixes this "echo chamber" effect by picking results iteratively using a two-part scoring formula:
1. Relevance: How closely a candidate item matches the search query.
2. Diversity: How different a candidate item is from the items already chosen

__Multi-query retrieval:__ Generate multiple phrasings of the user's question and retrieve for each. Useful when the user's phrasing doesn't match how the document was written.

```python
from langchain.retrievers import MultiQueryRetriever
retriever = MultiQueryRetriever.from_llm(retriever=base_retriever, llm=llm)
```

In [7]:
# Import modules for loading data from a vector store and use the same embeddings model to ensure compatibility
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Instantiate the embeddings model with the specified model name
embeddings = HuggingFaceEmbeddings(model_name="nomic-ai/nomic-embed-text-v1.5", model_kwargs = {'trust_remote_code': True})

# Load the vector store from disk for RAG tasks. 
# The `allow_dangerous_deserialization=True` parameter is used to allow loading of potentially unsafe data, which can be necessary in some cases but should be used with caution.
vector_store = FAISS.load_local("faiss_index", embeddings,
                                allow_dangerous_deserialization=True)

retriever = vector_store.as_retriever(
    search_type="similarity",  # or "mmr"
    search_kwargs={"k": 4},    # return top-4 chunks (Default: 4)
)

docs = retriever.invoke("What is BreakDown?") # Retrieve relevant documents based on the query

<All keys matched successfully>


#### Step 2 — Prompt for RAG context

Use ChatPromptTemplate to induce the context of the document in your query to LLM.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer using the context below. If the answer is not in the context, say "I don't know".
Context: {context}
Question: {question}""")

#### Step 3 — Answer with a RAG chain

Combine the retriever and LLM into an LCEL that fetches context and answers in one call.

In [ ]:
#from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


# Instantiate the ChatOpenAI model with the specified model name and temperature
#model = ChatOpenAI(model='gpt-4o-mini', temperature=0)
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.5
)

# Chain without RAG
chain = model | StrOutputParser()
response = chain.invoke("What is BreakDown?")
print(f'Without RAG: {response}')

# Chain with RAG
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt 
    | model 
    | StrOutputParser()
)
rag_response = rag_chain.invoke("What is BreakDown?")
print(f'With RAG: {rag_response}')

BadRequestError: Error code: 400 - {'error': {'message': 'The model `llama3-8b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}

<a href="https://colab.research.google.com/drive/1R6fYYtk_kTUrFj_WWCskLl954BgZHkQP?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

In [ ]:
#The getpass module is a built-in library in Python that lets you prompt a user for sensitive input—like a password—without echoing or displaying the characters on the screen
import getpass
import os

The main difference between bm25s and rank-bm25 is speed and architectural design. While both are pure Python packages used for keyword-based search ranking, bm25s was built specifically to overcome the severe performance bottlenecks of the older rank-bm25 library.
Here is a direct comparison of their differences:

Key Differences Overview


Feature	rank-bm25	bm25s
Performance (Speed)	🐢 Slow. Query speeds drop drastically on large datasets.	⚡ Extremely Fast. Up to 500x faster than rank-bm25.
Core Architecture	Computes relevance scores dynamically during query time.	Precomputes ("eagerly scores") scores during indexing.
Storage & Memory	Uses basic NumPy arrays or native Python loops.	Uses SciPy sparse matrices (csc_matrix) & optional disk memory-mapping (mmap).
BM25 Variants	Primarily limited to standard Okapi BM25 and a few basic variants.	Natively supports Lucene, Robertson, ATIRE, BM25L, and BM25+.
Feature Flexibility	Allows you to dynamically change parameters (like k₁) during search.	Requires saving/re-indexing if core parameters change, but supports Numba JIT and Hugging Face integration.


Why bm25s is so much faster

1. Eager Sparse Scoring: rank-bm25 has to do the heavy math when you submit a query. In contrast, bm25s eagerly calculates the BM25 scores for all document tokens during the index phase and stores them in SciPy sparse matrices. When you query, it simply slices the relevant indices and sums them up.
2. Memory Efficiency: If you have over a million documents, rank-bm25 will slow to a crawl or consume immense amounts of RAM. bm25s supports memory-mapping (mmap=True), meaning it can stream data directly from disk to keep RAM usage minimal.
3. Advanced Optimization: bm25s leverages fast O(n) top-k selection (Quickselect) and optional Numba JIT compilation to process thousands of queries per second (QPS).

Which one should you use?

• Use bm25s if you are working with large text datasets, building a Retrieval-Augmented Generation (RAG) system, or need production-grade performance without setting up heavy Java-based infrastructure like Elasticsearch.
• Use rank-bm25 only if you have a tiny corpus (e.g., hundreds of documents) where performance does not matter, or you need to constantly experiment with varying k₁ and b hyperparameters on the fly without re-indexing your data.
Would you like to see a code implementation example showing how to index and query data using bm25s?

show running denominator in 3rd section as exponential powers and multiplication of correction factor as well for better illustration. and remove "why does the rescaling work". Also rename the 3rd section as: Online scan like something. In the final takeaway: Rescaling- pretending that maximum was 7 all along. Also show GPU HBM memory traffic as high visually

how you give toke id to a sentence to generate embedding vector in rag pipeline?
11 October 2026
In a RAG (Retrieval-Augmented Generation) pipeline, you do not manually assign token IDs to a sentence; instead, the model's built-in tokenizer converts the text into token IDs, which are then passed through an embedding layer and pooled into a single sentence vector.
You typically invoke a high-level library (such as SentenceTransformers or langchain) that handles this entire chain automatically.

Step-by-Step Pipeline Flow

1. Tokenization (Text → Token IDs):
	• The text is split into subwords, words, or characters using the model's specific tokenizer (e.g., Byte Pair Encoding or WordPiece).
	• Each unit maps to a unique integer ID from the model's vocabulary.
2. Embedding Lookup (Token IDs → Token Vectors):
	• Each token ID acts as an index to look up a starting vector in the model's embedding matrix.
3. Transformer Processing & Pooling (Token Vectors → Sentence Vector):
	• The token vectors pass through neural network layers to capture context and relationships.
	• A pooling layer (such as mean pooling or CLS pooling) combines all token embeddings into one fixed-size vector representing the entire sentence.

Python Code Example

Using the SentenceTransformers library, the framework manages tokenization and vector generation behind the scenes:
python
from sentence_transformers import SentenceTransformer

# 1. Load the model (which includes the tokenizer and embedding layers)
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Input your sentence/chunk
document = (
    "Photosynthesis is the process by which plants convert sunlight into energy"
)

# 3. Inspect tokens manually (optional step)
tokenizer = model.tokenizer
tokens = tokenizer.tokenize(document)
print("Tokens:", tokens)

# 4. Generate the final sentence embedding vector directly
embedding = model.encode(document)
print(f"Embedding shape: {embedding.shape}")  # e.g., (384,)
Use code with caution.
If you prefer to inspect the raw token IDs explicitly before encoding:
python
# Convert text to token IDs directly via the tokenizer
token_ids = tokenizer.encode(document)
print("Token IDs:", token_ids)

# You can pass token IDs or text back to the model/tokenizer pipeline
Use code with caution.
If you are using a specific framework like OpenAI API, Hugging Face transformers, or LangChain instead of SentenceTransformers, let me know so I can provide the exact code for that setup.
AI responses may include mistakes. Learn more
• 
Medium
The Complete Guide to Embeddings and RAG
Step-by-Step Process Tokenizer (converts text → token IDs) # 2. Embedding layers (converts token IDs → contextualized embeddings) ...
• 
YouTube·CodeCraft Academy
From Words to Vectors: The Magic of AI Explained
every embedding model every retrieval system and every rag pipeline ultimately needs raw text converted into numbers into a single...
2:59
• 
LinkedIn
Understanding RAG model embedding matrix and token IDs
Understanding RAG model embedding matrix and token IDs AI Engineer | Python Developer | Open-Source Contributor | Built & Publishe...
Show all

You said: sentences has many combinations of words how embedding and tokenizer than generate such huge combinations

sentences has many combinations of words how embedding and tokenizer than generate such huge combinations
11 October 2026
Tokenizers and embedding models handle an infinite variety of sentences by breaking language down into subwords instead of whole words. This approach relies on a fixed, manageable vocabulary that can be combined to represent any text imaginable.
The combination of words is handled by the Transformer architecture, which scores how those words interact with each other in context.

1. The Tokenizer: Breaking Words into Subwords

If a tokenizer tried to store every unique word in existence, it would need a massive vocabulary and would break whenever it encountered a typo or a new slang term.
Instead, modern tokenizers use algorithms like Byte-Pair Encoding (BPE) or WordPiece to break text into common word pieces.
• A tokenizer typically has a fixed vocabulary size of 30,000 to 100,000 tokens.
• Regular words like plant get their own unique token ID.
• Rare, complex, or new words are split into pieces. For example, the word Photosynthesis might be broken into Photo, syn, the, and sis.
Because everything can eventually be broken down into individual letters or symbols if necessary, a tokenizer never encounters an "unknown" word. It can turn absolutely any combination of characters into a sequence of token IDs from its fixed list.

2. The Embedding Layer: Mapping IDs to Base Vectors

Once the tokenizer turns a sentence into a list of token IDs, those IDs point to a giant lookup table called the Embedding Matrix.
If a model has a vocabulary of 30,000 tokens and an embedding size of 768, this matrix is simply a list of 30,000 unique vectors. Each vector represents the basic, isolated meaning of that specific subword token.

3. The Transformer: Mixing Vectors for Infinite Variety

The real magic that solves the "huge combinations" problem happens in the Transformer layers, specifically through a mechanism called Self-Attention.
1. Contextual Enrichment: When token vectors enter the Transformer, they are completely isolated. As they pass through the network, the attention mechanism allows every token to "look" at every other token in the sentence.
2. Dynamic Shifting: The vector for the word bank changes completely depending on whether the surrounding tokens are river and water, or money and account.
3. Pooling: Finally, all of these context-rich token vectors are mathematically averaged or combined (usually via Mean Pooling) into one single vector.

The Analogy: The "RGB" Color Spectrum

Think of token embeddings like Red, Green, and Blue (RGB) values.
• You only need three primary colors (a small, fixed vocabulary).
• By mixing different amounts of red, green, and blue together (Self-Attention), you can create 16.7 million unique colors (infinite sentence combinations).
An embedding model does the exact same thing, but instead of 3 colors, it mixes hundreds of thousands of subword concepts across 768 or more dimensions to capture the exact mathematical shade of your sentence's meaning.
To help you dive deeper, tell me:
• Do you want to see how Self-Attention mathematically shifts these vectors?
• Are you trying to optimize your RAG pipeline's chunking strategy to handle long combinations better?
AI responses may include mistakes. Learn more

is maximum context size has any effect on chunking?

Yes, the model's maximum context size has a direct, critical effect on your chunking strategy in a RAG pipeline. It establishes the absolute boundary for how much text your embedding model and your LLM can safely ingest.
However, you must look at context size in two distinct parts of the pipeline: the Embedding Model and the Generation LLM.

1. The Embedding Model Context Limit (The Hard Wall)

Every embedding model has a maximum input limit, typically 512 tokens (for older or lighter models like all-MiniLM-L6-v2) or 8,192 tokens (for modern models like text-embedding-3-small or bge-large-en-v1.5).
• What happens if your chunk exceeds this limit? If your chunk size is set to 1,000 tokens but your embedding model only accepts 512, the tokenizer will silently truncate everything past token 512.
• The RAG Failure: The information in the second half of that text is completely lost to your vector database. The system will never be able to retrieve it because it was never vectorized.
• The Rule: Your chunk size must always be smaller than or equal to your embedding model's maximum context size.

2. The Generation LLM Context Limit (The Packing Problem)

Even if your embedding model can handle massive chunks (e.g., 8,192 tokens), your LLM's context window (e.g., 128k or 1M tokens) dictates how many retrieved chunks you can pass to it at generation time.
• Dilution vs. Completeness:
	• Large Chunks (e.g., 2,000 tokens): You preserve excellent context within the chunk, but you can only feed a few of them to the LLM before crowding out its prompt space or hitting limits. It can also cause "needle in a haystack" problems where the LLM misses details buried in the middle of a massive chunk.
	• Small Chunks (e.g., 200 tokens): You can pack dozens of highly specific, diverse snippets into the LLM's context window. However, if the answer requires connecting concepts that span across pages, small chunks might fragment the meaning too much.

Strategy: The "Golden Zone" for Chunking

To balance these context sizes, most RAG pipelines optimize chunking using these design patterns:
• Target a Sweet Spot: A very common, safe default is a chunk size of 256 to 512 tokens with a 10% to 20% overlap (e.g., 50 tokens). This fits cleanly inside almost all modern embedding models and allows you to retrieve the top 5 to 10 most relevant chunks without overwhelming the LLM.
• Parent-Child Retrieval: If your data requires massive context but precise retrieval, use a split strategy. Chunk your data into tiny Child chunks (128 tokens) for the embedding model to find easily, but map them to larger Parent chunks (1,500 tokens). When a child matches, pass the larger parent chunk to the LLM.
To tailor this to your exact system, let me know:
• What embedding model and LLM are you currently using or planning to use?
• What type of data are you chunking (e.g., short customer support tickets, long legal contracts, or code repositories)?